# Etapa 2 — Pipeline de Tratamento / ETL

- **Tema:** Acidentes nas rodovias federais do Paraná (2021–2025): onde, quando e por que acontecem os acidentes mais graves
- **Equipe:** Alexandre, Laerto, Klaus e Kauan
- **Fonte:** PRF — Dados Abertos, *Acidentes (agrupados por ocorrência)*, um CSV (em ZIP) por ano — <https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf>

Este notebook demonstra, passo a passo, o tratamento feito pelo pipeline `src/etl.py`.
Cada etapa é uma função do pipeline; aqui ela é chamada isoladamente para mostrar **o estado antes, a regra aplicada e o estado depois**.
O mesmo resultado é obtido de uma vez com `python src/etl.py`.

| # | Etapa | Critério da avaliação |
|---|---|---|
| 1 | Extração dos 5 anos | — |
| 2 | Padronização de texto e nomes de colunas | normalização estrutural |
| 3 | Filtragem (PR, 2021–2025) | filtragens pertinentes ao tema |
| 4 | Tratamento de nulos (incl. nulos "disfarçados") | resolução de valores nulos |
| 5 | Remoção de duplicados (3 níveis) | resolução de duplicados |
| 6 | Tipagem | tipagem correta de variáveis |
| 7 | Normalização estrutural (rótulos, campo multivalorado) | normalização estrutural |
| 8 | Variáveis derivadas para as hipóteses | filtragens pertinentes ao tema |
| 9 | Validação automática | — |
| 10 | Carga (Parquet + CSV) | — |

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / "src"))
import etl

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
etl.LOG.clear()

## 1. Extração

Os ZIPs são baixados do Google Drive da PRF (os IDs estão em `etl.DRIVE_IDS`, tirados da página oficial) para `data/raw/`.
Os CSVs usam **`;` como separador e codificação Latin-1**.

Decisão importante: tudo é lido **como texto** (`dtype=str`) e **sem conversão automática de nulos** (`keep_default_na=False`).
Assim nenhum valor é convertido "no escuro" pelo pandas: cada nulo e cada tipo é decidido de forma explícita nas etapas seguintes.

In [2]:
etl.baixar()
bruto = etl.extrair()
print(f"{len(bruto):,} acidentes no Brasil, {bruto.shape[1]} colunas".replace(",", "."))
bruto.groupby("ano_arquivo").size().rename("acidentes (Brasil)").to_frame().T

342.670 acidentes no Brasil. 31 colunas


ano_arquivo,2021,2022,2023,2024,2025
acidentes (Brasil),64567,64606,67766,73202,72529


In [3]:
# Todas as colunas chegam como texto. Repare nos números com vírgula decimal (km, latitude, longitude).
print("Tipos:", bruto.dtypes.value_counts().to_dict())
bruto[["id", "data_inversa", "horario", "uf", "br", "km", "latitude", "longitude", "condicao_metereologica", "tracado_via"]].head(5)

Tipos: {dtype('O'): 30, dtype('int64'): 1}


,id,data_inversa,horario,uf,br,km,latitude,longitude,condicao_metereologica,tracado_via
0,331730,2021-01-01,05:30:00,SP,116,453,"-24,56168114","-47,86752174",Nublado,Curva
1,331804,2021-01-01,08:05:00,AM,174,937,"-2,508068","-60,036434",Céu Claro,Reta
2,331815,2021-01-01,10:10:00,CE,222,"128,5","-3,707626","-39,623509",Céu Claro,Curva
3,331823,2021-01-01,12:30:00,RJ,493,18,"-22,6687122","-43,01862696",Nublado,Curva
4,331843,2021-01-01,14:40:00,RJ,393,252,"-22,43422023","-43,77561487",Nublado,Reta


## 2. Padronização de texto e nomes de colunas

- normalização Unicode (NFC) e remoção de espaços nas pontas / espaços repetidos — evita que o mesmo valor com um espaço a mais vire outra categoria (na base aparece, por exemplo, município com espaço duplo);
- nomes de colunas: `condicao_metereologica` → `condicao_meteorologica` (erro de grafia da fonte), `data_inversa` → `data`, `uso_solo` → `area_urbana` (na fonte, *Sim* = área urbana).

In [4]:
df = etl.padronizar_texto(bruto)
mudou = pd.DataFrame({c: bruto[c] != df[etl.RENOMEAR.get(c, c)] for c in bruto.columns if bruto[c].dtype == object})
print("Células alteradas:", int(mudou.values.sum()))
exemplos = [(c, repr(v)) for c in mudou.columns for v in bruto.loc[mudou[c], c].unique()[:3]]
pd.DataFrame(exemplos, columns=["coluna", "valor original (repr)"])

Células alteradas: 50


,coluna,valor original (repr)
0,municipio,'SAO SEBASTIAO DE LAGOA DE ROCA'


## 3. Filtragem pertinente ao tema

Recorte da Etapa 1: **acidentes ocorridos no Paraná, de 2021 a 2025**.
O filtro usa a coluna `uf`, que é a **UF do local do acidente**, e não a da delegacia que atendeu:
alguns acidentes no PR (divisa) são atendidos por delegacias de SC e SP e devem continuar na base.

In [5]:
pr = etl.filtrar(df)
print(f"Brasil: {len(df):,} -> Paraná 2021–2025: {len(pr):,} acidentes".replace(",", "."))
print("Acidentes no PR atendidos por delegacia de outra UF (mantidos):",
      int((~pr['delegacia'].str.endswith('-PR') & ~pr['delegacia'].isin(etl.SENTINELAS_NULO)).sum()))
pr.groupby("ano_arquivo").size().rename("acidentes no PR").to_frame().T

Brasil: 342.670 -> Paraná 2021–2025: 37.062 acidentes
Acidentes no PR atendidos por delegacia de outra UF (mantidos): 237


ano_arquivo,2021,2022,2023,2024,2025
acidentes no PR,7346,7393,7110,7583,7630


## 4. Tratamento de valores nulos

Na base da PRF quase não existe célula vazia: **o nulo vem disfarçado** de texto.
O primeiro passo é encontrar esses valores-sentinela e convertê-los em nulo de verdade:

| Sentinela | Onde aparece | Significado |
|---|---|---|
| `Ignorado` | `condicao_meteorologica` | clima não registrado |
| `Não Informado` | `sentido_via` | sentido não registrado |
| `NA`, `N/A` | `regional`, `delegacia`, `uop`, `classificacao_acidente`* | sem informação |
| `br = 0` (e `km = 0`) | `br`, `km` | não existe BR-000: rodovia não informada |

\* `classificacao_acidente = NA` aparece 5 vezes no Brasil, nenhuma no PR — a regra fica no pipeline mesmo assim.

**Perfil de nulos do recorte PR, antes do tratamento:**

In [6]:
etl.perfil_nulos(pr)

,coluna,nulos_reais,sentinelas,total,pct
0,br,0,{'0': 70},70,0.19
1,sentido_via,0,{'Não Informado': 70},70,0.19
2,condicao_meteorologica,0,{'Ignorado': 487},487,1.31
3,regional,0,{'NA': 1},1,0.00
4,delegacia,0,{'NA': 1},1,0.00
5,uop,0,"{'N/A': 11, 'NA': 1}",12,0.03


**Regra adotada para cada caso** — o princípio foi *não inventar informação que a análise vá usar*:

| Coluna | Regra | Por quê |
|---|---|---|
| `classificacao_acidente` | recalculada a partir de `mortos`/`feridos` | é derivável sem ambiguidade |
| `regional`/`delegacia`/`uop` | copia o trio do acidente mais próximo (km) na **mesma BR e município**; sem vizinho → `"Não informado"` | a PRF atribui a unidade por trecho; coluna administrativa, não entra nas hipóteses |
| `condicao_meteorologica`, `sentido_via` | categoria explícita `"Não informado"` | imputar clima seria inventar dado justamente da hipótese 4; a linha é mantida porque as outras colunas são válidas, e esses casos saem só da análise que usa a coluna |
| `br`, `km` | mantidos como nulo + flag `localizacao_valida = False` | 70 ocorrências fora de trecho de BR; entram nas hipóteses 2–4 e saem só da hipótese 1 (trechos) |

Nenhuma linha é descartada por causa de nulo.

In [7]:
tratado = etl.tratar_nulos(pr)
print(json.dumps(etl.LOG[-1]["regras"], ensure_ascii=False, indent=1))

{
 "classificacao_acidente": "0 derivadas de mortos/feridos",
 "regional/delegacia/uop": "12 linha(s): 1 pelo acidente mais próximo na mesma BR/município, 11 célula(s) -> 'Não informado'",
 "condicao_meteorologica": "487 -> categoria 'Não informado' (excluídas só na hipótese que usa a coluna)",
 "sentido_via": "70 -> categoria 'Não informado' (excluídas só na hipótese que usa a coluna)",
 "br/km": "70 mantidos como nulo + flag localizacao_valida=False"
}


In [8]:
# Depois: o único nulo que sobra é o intencional (br/km fora da malha federal), sinalizado pela flag
restantes = tratado.isna().sum()
display(restantes[restantes > 0].rename("nulos").to_frame())
print("Linhas com localizacao_valida = False:", int((~tratado["localizacao_valida"]).sum()))

,nulos
br,70
km,70


Linhas com localizacao_valida = False: 70


In [9]:
# O único caso de imputação por vizinho: id 619297 (Ponta Grossa, BR-376 km 475) estava com regional/delegacia/uop = 'NA'
pd.concat([pr.loc[pr["id"] == "619297", ["id", "municipio", "br", "km", "regional", "delegacia", "uop"]].assign(estado="antes"),
           tratado.loc[tratado["id"] == "619297", ["id", "municipio", "br", "km", "regional", "delegacia", "uop"]].assign(estado="depois")])

,id,municipio,br,km,regional,delegacia,uop,estado
21848,619297,PONTA GROSSA,376,475,NA,NA,NA,antes
21848,619297,PONTA GROSSA,376,475,SPRF-PR,DEL03-PR,UOP01-DEL03-PR,depois


In [10]:
# A regra de classificação, testada nos 5 registros do Brasil que vêm com 'NA'
na_br = df[df["classificacao_acidente"] == "NA"][["id", "uf", "mortos", "feridos", "classificacao_acidente"]].copy()
m, f = na_br["mortos"].astype(int), na_br["feridos"].astype(int)
na_br["classificacao_derivada"] = ["Com Vítimas Fatais" if a > 0 else "Com Vítimas Feridas" if b > 0 else "Sem Vítimas" for a, b in zip(m, f)]
na_br

,id,uf,mortos,feridos,classificacao_acidente,classificacao_derivada
1,331804,AM,0,1,NA,Com Vítimas Feridas
64567,405151,PI,0,1,NA,Com Vítimas Feridas
129175,496590,MT,0,1,NA,Com Vítimas Feridas
196939,571789,ES,0,1,NA,Com Vítimas Feridas
270142,652519,CE,1,1,NA,Com Vítimas Fatais


## 5. Duplicados

Verificados em três níveis:

1. **`id` repetido** — o `id` é a chave da ocorrência na PRF;
2. **linha idêntica exceto o `id`** — mesmo registro lançado duas vezes;
3. **duplicado lógico** — mesmo instante, mesmo ponto (km **e** coordenadas) e mesmas contagens de pessoas e veículos.
   Ficam de fora da chave, de propósito, campos de *percepção/registro* (clima, traçado, delegacia),
   que mudam quando dois agentes ou duas delegacias registram o mesmo acidente.

Em todos os casos fica o menor `id` (o primeiro registro). Os removidos vão para `data/processed/auditoria_duplicados_removidos.csv`.

In [11]:
sem_dup, removidos = etl.remover_duplicados(tratado)
{k: v for k, v in etl.LOG[-1].items() if k not in ("etapa",)}

{'linhas_antes': 37062,
 'linhas_depois': 37059,
 'id_repetido': 0,
 'linha_identica': 0,
 'duplicado_logico': 3}

Nenhum `id` repetido nem linha idêntica. A busca por **mesma data, hora, BR e km** encontra pares suspeitos,
e cada um foi analisado campo a campo antes de definir a chave final:

In [12]:
def diferencas(a, b):
    x, y = tratado[tratado["id"] == a].iloc[0], tratado[tratado["id"] == b].iloc[0]
    return {c: f"{x[c]}  |  {y[c]}" for c in tratado.columns if x[c] != y[c] and c not in ("id",)}

pares = [("361574", "361577"), ("411196", "411197"), ("443372", "450136"), ("486992", "487025"), ("353903", "353904")]
linhas = []
for a, b in pares:
    d = diferencas(a, b)
    linhas.append({"par de ids": f"{a} / {b}", "campos que diferem": "; ".join(f"{k}: {v}" for k, v in d.items()),
                   "decisão": "REMOVIDO (duplicado)" if b in set(removidos["id"]) else "mantidos (acidentes distintos)"})
pd.DataFrame(linhas)

,par de ids,campos que diferem,decisão
0,361574 / 361577,condicao_meteorologica: Sol | Nublado; area_urbana: Não | Sim,REMOVIDO (duplicado)
1,411196 / 411197,condicao_meteorologica: Sol | Céu Claro; tracado_via: Reta | A...,REMOVIDO (duplicado)
2,443372 / 450136,tracado_via: Reta | Interseção de Vias; delegacia: DEL01-PR | ...,REMOVIDO (duplicado)
3,486992 / 487025,"pessoas: 1 | 4; ilesos: 1 | 4; latitude: -24,80531402 | -24,...",mantidos (acidentes distintos)
4,353903 / 353904,classificacao_acidente: Com Vítimas Feridas | Sem Vítimas; traca...,mantidos (acidentes distintos)


- **361574/361577** e **411196/411197**: ids seguidos, mesmo ponto exato e mesmas contagens; só muda a percepção de clima/traçado → o mesmo acidente registrado duas vezes.
- **443372/450136**: tudo igual, mas registrado por **duas delegacias** diferentes → duplicado.
- **486992/487025** (Tibagi, chuva): coordenadas a ~7 km uma da outra e 1 × 4 ocupantes → **dois veículos diferentes** aquaplanando no mesmo trecho. Uma primeira versão da chave (sem coordenadas) removia este par por engano; a análise caso a caso corrigiu a regra.
- **353903/353904**: vítimas e coordenadas diferentes → acidentes distintos.

## 6. Tipagem

| Coluna(s) | De | Para | Observação |
|---|---|---|---|
| `id` | texto | `int64` | a fonte traz id em notação científica (`6e+05` = 600000); conversão conferida (inteiro e único) |
| `data`, `horario` | texto | `datetime64`, `time` + nova `data_hora` | combinadas para permitir análise por hora |
| `km`, `latitude`, `longitude` | texto com **vírgula** decimal | `Float64` | `"546,2"` → `546.2` |
| `br` | texto | `Int16` (aceita nulo) | |
| contagens (`mortos`, `feridos`, ...) | texto | `int16` | |
| `area_urbana` | "Sim"/"Não" | booleano | |
| `dia_semana`, `fase_dia`, `classificacao_acidente` | texto | categoria **ordenada** | permite ordenar segunda→domingo e Sem vítimas < Feridas < Fatais |
| demais textos | texto | `category` | economia de memória e domínio fechado |

In [13]:
# O caso do id em notação científica
print(sem_dup.loc[~sem_dup["id"].str.fullmatch(r"\d+"), ["id", "data", "horario", "municipio"]])
tipado = etl.tipar(sem_dup)
print("\nDepois:", tipado.loc[tipado["id"] == 600000, ["id", "data", "horario", "municipio"]].to_string(index=False))

          id        data   horario             municipio
25809  6e+05  2024-05-12  12:45:00  SAO JOSE DOS PINHAIS

Depois:     id       data  horario            municipio
600000 2024-05-12 12:45:00 SAO JOSE DOS PINHAIS


In [14]:
cols = ["id", "data", "horario", "data_hora", "br", "km", "latitude", "mortos", "area_urbana", "dia_semana", "classificacao_acidente", "causa_acidente"]
antes_t = sem_dup.reindex(columns=cols).dtypes.astype(str)
antes_t["data_hora"] = "(não existia)"
comparacao = pd.DataFrame({"antes": antes_t, "depois": tipado[cols].dtypes.astype(str),
                           "exemplo depois": [str(tipado[c].iloc[1]) for c in cols]})
comparacao.loc[comparacao["depois"].str.startswith("category"), "depois"] = [
    "category (ordenada)" if tipado[c].cat.ordered else "category" for c in comparacao.index[comparacao["depois"].str.startswith("category")]]
comparacao

,antes,depois,exemplo depois
id,object,int64,331920
data,object,datetime64[ns],2021-01-01 00:00:00
horario,object,object,20:40:00
data_hora,(não existia),datetime64[ns],2021-01-01 20:40:00
br,object,Int16,116
km,object,Float64,59.0
latitude,object,Float64,-25.30335724
mortos,object,int16,1
area_urbana,object,boolean,True
dia_semana,object,category (ordenada),sexta-feira


In [15]:
mb = lambda d: d.memory_usage(deep=True).sum() / 1e6
print(f"Memória: {mb(sem_dup):.1f} MB (texto) -> {mb(tipado):.1f} MB (tipado)")

Memória: 76.9 MB (texto) -> 5.2 MB (tipado)


## 7. Normalização estrutural

**a) Rótulos de causa que mudaram entre os anos.** A PRF trocou a grafia de algumas causas ao longo do período.
Sem harmonizar, a mesma causa aparece como três categorias e as séries por ano ficam quebradas:

In [16]:
afetadas = list(etl.MAPA_CAUSAS) + sorted(set(etl.MAPA_CAUSAS.values()))
pd.crosstab(tipado["causa_acidente"].astype(str), tipado["ano_arquivo"]).loc[afetadas]

ano_arquivo,2021,2022,2023,2024,2025
causa_acidente,,,,,
Transitar no acostamento,41,17,0,0,0
Ingestão de álcool e/ou substâncias psicoativas pelo pedestre,43,30,25,0,0
Ingestão de álcool ou de substâncias psicoativas pelo pedestre,21,12,9,0,0
Pedestre - Ingestão de álcool/ substâncias psicoativas,0,0,0,18,14
Transitar no Acostamento,0,0,16,38,35


In [17]:
normal, ponte = etl.normalizar(tipado)
alvos = sorted(set(etl.MAPA_CAUSAS.values()))
display(pd.crosstab(normal["causa_acidente"].astype(str), normal["ano_arquivo"]).loc[alvos])
print("Causas distintas:", etl.LOG[-1]["causas_distintas"])

ano_arquivo,2021,2022,2023,2024,2025
causa_acidente,,,,,
Pedestre - Ingestão de álcool/ substâncias psicoativas,64,42,34,18,14
Transitar no Acostamento,41,17,16,38,35


Causas distintas: 76 -> 73


**b) Campo multivalorado sem ordem: `tracado_via`.** Um acidente pode ter vários elementos de traçado, separados por `;`,
em qualquer ordem — `"Reta;Declive"` e `"Declive;Reta"` são a mesma coisa, mas contam como categorias diferentes.
Isso fere a primeira forma normal (um valor atômico por célula). O tratamento:

1. forma canônica (elementos únicos e em ordem alfabética);
2. uma coluna booleana por elemento (`tracado_curva`, `tracado_declive`, ...) para filtrar direto no dashboard;
3. tabela-ponte `acidente_tracado.csv` (uma linha por acidente × elemento), que é a forma normalizada para análise relacional (Power BI / SQL).

In [18]:
ex = tipado["tracado_via"].astype(str)
exemplo = ex[ex.isin(["Reta;Declive", "Declive;Reta"])].value_counts()
display(exemplo.rename("antes").to_frame().assign(depois=[etl._tracado_canonico(i) for i in exemplo.index]))
print("Combinações distintas:", etl.LOG[-1]["tracado_combinacoes"])

,antes,depois
tracado_via,,
Reta;Declive,1076,Declive;Reta
Declive;Reta,745,Declive;Reta


Combinações distintas: 395 -> 166


In [19]:
display(ponte.head(6))
normal.filter(like="tracado_").drop(columns="tracado_via").sum().sort_values(ascending=False).rename("acidentes").to_frame().T

,id,elemento
0,331855,Curva
1,331920,Curva
2,331920,Declive
3,332106,Curva
4,332264,Reta
5,332449,Reta


,tracado_reta,tracado_curva,tracado_declive,tracado_aclive,tracado_intersecao_de_vias,tracado_retorno_regulamentado,tracado_rotatoria,tracado_viaduto,tracado_em_obras,tracado_ponte,tracado_desvio_temporario,tracado_tunel
acidentes,24660,8466,4483,3276,3131,880,713,558,480,205,130,9


Mantido como está, e registrado como limitação: o tipo **`Colisão lateral`** só existe em 2021 (71 casos).
Depois a PRF passou a separar *mesmo sentido* e *sentido oposto*, e não há como saber qual era cada caso de 2021.

## 8. Variáveis derivadas para as hipóteses

| Variável | Regra | Hipótese |
|---|---|---|
| `fatal` | `mortos > 0` | H1–H4 |
| `grave` | `mortos > 0` ou `feridos_graves > 0` | H1–H4 |
| `trecho_10km` | BR + faixa de 10 km (ex.: `BR-277 km 50–60`); nulo se `localizacao_valida = False` | H1 — concentração em trechos |
| `noite` | `fase_dia` ∈ {Plena Noite, Anoitecer} | H2 |
| `fim_de_semana` | sábado ou domingo | H2 |
| `ano`, `mes`, `hora` | extraídos de `data_hora` | H2 / séries temporais |
| `chuva` | `condicao_meteorologica` ∈ {Chuva, Garoa/Chuvisco, Granizo} | H4 |
| `pista_molhada_proxy` | `chuva` **ou** causa ∈ {Acúmulo de água sobre o pavimento, Pista escorregadia, Chuva} | H4 |
| `total_envolvidos` | mortos + feridos + ilesos + ignorados | consistência |

**Ajuste em relação à Etapa 1:** o conjunto *agrupado por ocorrência* **não tem mais a coluna "condição da pista"**.
Para a hipótese 4 usamos o clima registrado e as causas ligadas a pista molhada (`pista_molhada_proxy`).

**Sobre `pessoas`:** em 2.650 linhas `pessoas` é menor que a soma de mortos + feridos + ilesos + ignorados.
A diferença aparece só quando há `ignorados`, ou seja, a PRF não conta em `pessoas` os envolvidos não identificados.
Não é erro a corrigir: a coluna original é mantida e `total_envolvidos` traz o total.

In [20]:
final = etl.derivar(normal)
resumo = pd.Series({
    "acidentes": len(final), "fatais": int(final["fatal"].sum()), "graves (fatal ou ferido grave)": int(final["grave"].sum()),
    "à noite": int(final["noite"].sum()), "no fim de semana": int(final["fim_de_semana"].sum()),
    "com chuva": int(final["chuva"].sum()), "pista molhada (proxy)": int(final["pista_molhada_proxy"].sum()),
    "trechos de 10 km distintos": final["trecho_10km"].nunique()}).rename("quantidade").to_frame()
resumo

,quantidade
acidentes,37059
fatais,2502
graves (fatal ou ferido grave),9851
à noite,15493
no fim de semana,12221
com chuva,6444
pista molhada (proxy),6553
trechos de 10 km distintos,416


In [21]:
print("pessoas != soma das categorias:", int((final["pessoas"] != final["total_envolvidos"]).sum()))
print("...e destes, quantos têm ignorados > 0:", int(((final["pessoas"] != final["total_envolvidos"]) & (final["ignorados"] > 0)).sum()))

pessoas != soma das categorias: 2650
...e destes, quantos têm ignorados > 0: 2650


## 9. Validação automática

Regras que a base tratada tem de cumprir. Se alguma falhar, o pipeline para com erro — nada é gravado com problema.
(Foi esta etapa que pegou o `uop = "N/A"` que escapou da primeira versão do tratamento de nulos.)

In [22]:
checks = etl.validar(final)
pd.Series(checks).map({True: "OK", False: "FALHOU"}).rename("resultado").to_frame()

,resultado
id único,OK
anos só 2021–2025,OK
sem nulo nas colunas obrigatórias,OK
coordenadas dentro do PR,OK
feridos = leves + graves,OK
fatal <=> classificação 'Com Vítimas Fatais',OK
pessoas <= total_envolvidos (diferença = ignorados),OK
dia_semana coerente com a data,OK
br/km nulos só onde localizacao_valida=False,OK


## 10. Carga

| Arquivo | Conteúdo |
|---|---|
| `acidentes_pr_2021_2025.parquet` | base tratada com os tipos preservados (para Python/Streamlit) |
| `acidentes_pr_2021_2025.csv` | mesma base, UTF-8, ponto decimal (para Power BI/Excel) |
| `acidente_tracado.csv` | tabela-ponte acidente × elemento do traçado |
| `auditoria_duplicados_removidos.csv` | registros removidos e o motivo |
| `log_etl.json` | linhas antes/depois e regras de cada etapa |

In [23]:
salvo = etl.carregar(final, ponte, removidos)
print(f"Base final: {salvo.shape[0]:,} acidentes x {salvo.shape[1]} colunas".replace(",", "."))
funil = pd.DataFrame(etl.LOG)[["etapa", "linhas_antes", "linhas_depois"]]
funil["variação"] = funil["linhas_depois"] - funil["linhas_antes"]
funil["variação"] = funil["variação"].astype("Int64")
funil.loc[0, "variação"] = pd.NA
funil

Base final: 37.059 acidentes x 55 colunas


,etapa,linhas_antes,linhas_depois,variação
0,"1. Extração (5 CSVs, Brasil inteiro)",0,342670,<NA>
1,2. Padronização de texto e nomes,342670,342670,0
2,3. Filtragem: uf = PR e 2021–2025,342670,37062,-305608
3,4. Tratamento de nulos,37062,37062,0
4,5. Remoção de duplicados,37062,37059,-3
5,6. Tipagem,37059,37059,0
6,7. Normalização estrutural,37059,37059,0
7,8. Variáveis derivadas,37059,37059,0
8,9. Validação,37059,37059,0
9,10. Carga,37059,37059,0


In [24]:
# Conferência: o arquivo gravado volta com os mesmos tipos e sem nulos fora do esperado
relido = pd.read_parquet(etl.PROCESSED / "acidentes_pr_2021_2025.parquet")
print(relido.shape, "| nulos:", relido.isna().sum()[lambda s: s > 0].to_dict())
relido.head(3)

(37059, 55) | nulos: {'br': 70, 'km': 70, 'trecho_10km': 70}


,id,data,horario,data_hora,ano,mes,hora,dia_semana,fim_de_semana,fase_dia,noite,br,km,trecho_10km,localizacao_valida,municipio,latitude,longitude,sentido_via,area_urbana,...,feridos,ilesos,ignorados,veiculos,regional,delegacia,uop,ano_arquivo,tracado_aclive,tracado_curva,tracado_declive,tracado_desvio_temporario,tracado_em_obras,tracado_intersecao_de_vias,tracado_ponte,tracado_reta,tracado_retorno_regulamentado,tracado_rotatoria,tracado_tunel,tracado_viaduto
0,331855,2021-01-01,15:45:00,2021-01-01 15:45:00,2021,1,15,sexta-feira,False,Pleno dia,False,277,51.3,BR-277 km 50–60,True,SAO JOSE DOS PINHAIS,-25.59516,-48.907008,Decrescente,False,...,1,3,0,3,SPRF-PR,DEL01-PR,UOP05-DEL01-PR,2021,False,True,False,False,False,False,False,False,False,False,False,False
1,331920,2021-01-01,20:40:00,2021-01-01 20:40:00,2021,1,20,sexta-feira,False,Plena Noite,True,116,59.0,BR-116 km 50–60,True,CAMPINA GRANDE DO SUL,-25.303357,-48.943789,Crescente,True,...,0,1,1,3,SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2021,False,True,True,False,False,False,False,False,False,False,False,False
2,332106,2021-01-02,19:00:00,2021-01-02 19:00:00,2021,1,19,sábado,True,Anoitecer,True,277,309.1,BR-277 km 300–310,True,GUARAPUAVA,-25.307524,-51.210171,Crescente,False,...,4,1,2,4,SPRF-PR,DEL08-PR,UOP01-DEL08-PR,2021,False,True,False,False,False,False,False,False,False,False,False,False


## Resumo das decisões

- **Recorte:** 342.670 acidentes no Brasil → **37.062 no PR (2021–2025)** → **37.059** após remover 3 duplicados.
- **Nulos:** 641 nulos disfarçados encontrados (`Ignorado`, `Não Informado`, `NA`, `N/A`, `br = 0`). Nenhuma linha descartada; cada coluna com regra própria. Sobram só os 70 `br/km` fora da malha federal, sinalizados por flag.
- **Duplicados:** nenhum exato; 3 duplicados lógicos removidos depois de analisar caso a caso. Um par que parecia duplicado era, na verdade, de dois acidentes diferentes (Tibagi).
- **Tipagem:** de 30 colunas de texto para tipos numéricos, datas, booleanos e categorias ordenadas. Inclui a correção da vírgula decimal e do id em notação científica.
- **Normalização:** nomes de colunas corrigidos; 76 → 73 causas (grafias harmonizadas); `tracado_via` de 395 → 166 combinações, com colunas booleanas e tabela-ponte.
- **Pronto para a Etapa 3:** variáveis `fatal`, `grave`, `trecho_10km`, `noite`, `fim_de_semana`, `chuva` e `pista_molhada_proxy`, ligadas às hipóteses H1–H4.

**Limitações registradas:** a fonte não tem mais "condição da pista" (usamos proxy); `Colisão lateral` só em 2021; `pessoas` não conta os envolvidos ignorados.